<a href="https://colab.research.google.com/github/LPJanadriGit/predictive-diagnostics-capstone/blob/main/GenerateSyntheticTelematicsAndServiceWarrantyDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# 1. Force establish the directories in Google Colab environment
os.makedirs("data/rawdata", exist_ok=True)
os.makedirs("data/processeddata", exist_ok=True)
os.makedirs("data/datadictionaries", exist_ok=True)

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Global Configuration
vins = [f"1G1RC6E4XGU{random.randint(100000, 999999)}" for _ in range(3)]
records_per_vin = 100
total_records = len(vins) * records_per_vin

# -------------------------------------------------------------------------
# DATASET 4: SYNTHETIC TELEMATICS DATASET
# -------------------------------------------------------------------------
telematics_data = []

for vin in vins:
    base_time = datetime(2026, 1, 1, 8, 0, 0)
    for cycle in range(1, records_per_vin + 1):
        timestamp = base_time + timedelta(minutes=15 * cycle)

        speed = float(np.random.normal(75.0, 15.0))
        speed = max(0.0, min(speed, 160.0))

        engine_rpm = int(speed * 30 + np.random.normal(1200, 200))
        engine_rpm = max(800, min(engine_rpm, 5500))

        degradation_factor = np.exp(cycle / records_per_vin * 2) / np.exp(2)
        base_temp = 85.0 + (degradation_factor * 25.0)
        coolant_temp = float(base_temp + np.random.normal(0, 1.5))

        dtc_code = "None"
        if cycle > 75 and random.random() > 0.7:
            dtc_code = random.choice(["P0300", "P0128", "P0420"])
        elif cycle > 90:
            dtc_code = random.choice(["P0300", "P0171", "P0299"])

        telematics_data.append({
            "VIN": vin,
            "Timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
            "Cycle": cycle,
            "Vehicle_Speed_kmh": round(speed, 2),
            "Engine_RPM": engine_rpm,
            "Coolant_Temperature_C": round(coolant_temp, 2),
            "DTC_Code": dtc_code,
            "GPS_Latitude": round(42.3314 + np.random.normal(0, 0.05), 5),
            "GPS_Longitude": round(-83.0458 + np.random.normal(0, 0.05), 5)
        })

df_telematics = pd.DataFrame(telematics_data)

# -------------------------------------------------------------------------
# DATASET 5: SYNTHETIC SERVICE AND WARRANTY DATASET
# -------------------------------------------------------------------------
service_data = []

narrative_templates = {
    "P0300": [
        "Customer reported engine stumble. Found cylinder misfire logs. Replaced spark plugs and ignition coils.",
        "Misfire codes present on acceleration. Inspected fuel delivery rail; performed injector flush and recoded ECU."
    ],
    "P0128": [
        "Check engine light illuminated. Coolant temperature reading below target thresholds. Replaced stuck-open thermostat assembly.",
        "Thermal delay observed during driving profiles. Thermal sensor out of calibration limits; swapped out coolant sensor."
    ],
    "P0171": [
        "System running lean. Detected unmetered air leak past the mass airflow sensor. Replaced torn intake boot.",
        "Fuel trim metrics exceeding standard bounds. Swapped oxygen sensor and cleared internal adaptive matrices."
    ],
    "None": [
        "Routine scheduled fleet check up completed. Cleaned body panels and updated telematics system firmware parameters.",
        "Component parameters verified within target operational bands. No structural abnormalities identified during execution."
    ]
}

for vin in vins:
    vin_records = df_telematics[df_telematics["VIN"] == vin]
    max_cycle = vin_records["Cycle"].max()

    for _, row in vin_records.iterrows():
        cycle = row["Cycle"]
        current_dtc = row["DTC_Code"]

        rul = max_cycle - cycle
        anomaly_label = 1 if (rul <= 20 or current_dtc != "None") else 0

        if current_dtc != "None":
            fault_type = f"Subsystem_{current_dtc}_Failure"
        elif rul <= 15:
            fault_type = "End_of_Life_Degradation"
        else:
            fault_type = "Normal_Operation"

        notes_pool = narrative_templates.get(current_dtc, narrative_templates["None"])
        technician_note = random.choice(notes_pool)

        if fault_type != "Normal_Operation":
            warranty_cost = float(np.random.uniform(450.00, 2800.00))
            down_time_hours = float(np.random.exponential(scale=4.0) + 1.0)
        else:
            warranty_cost = 0.00
            down_time_hours = 0.00

        service_data.append({
            "VIN": vin,
            "Cycle": cycle,
            "RUL": rul,
            "Anomaly_Label": anomaly_label,
            "Fault_Type": fault_type,
            "Technician_Notes": technician_note,
            "Warranty_Cost_USD": round(warranty_cost, 2),
            "Downtime_Hours": round(down_time_hours, 1)
        })

df_service = pd.DataFrame(service_data)

# Save directly to the created folders inside Colab instance
df_telematics.to_csv("data/processeddata/synthetic_telematics_dataset.csv", index=False)
df_service.to_csv("data/processeddata/synthetic_service_warranty_dataset.csv", index=False)

print("✅ Success! Datasets generated and saved to Colab workspace folder tree.")
print(f"Telematics Shape: {df_telematics.shape} | Service Shape: {df_service.shape}")

✅ Success! Datasets generated and saved to Colab workspace folder tree.
Telematics Shape: (300, 9) | Service Shape: (300, 8)
